In [ ]:
from collections import deque
import os
import stat
import tkinter as tk
from typing import Generator, Tuple
from osxmetadata import OSXMetaData
from pathlib import Path
from tkinter import filedialog


def select_directory() -> str | None:
    """
    - 폴더 선택 다이얼로그 띄우기
    - 실제 플로우는 클라이언트 단에서 폴더를 선택하여 경로를 매개변수로 전달해줘야 하지만
    - 노트북에서 실제와 비슷한 플로우로 테스트를 하기 위한 용도로 구현
    """
    root = tk.Tk()
    root.withdraw()

    path = filedialog.askdirectory(initialdir=str(Path.home()))

    root.destroy()

    if path:
        return path
    else:
        return None


def _process_entry(
    entry: os.DirEntry[str],
    stack: deque[Path],
    follow_symlinks: bool = False,
) -> Generator[Tuple[Path, os.stat_result]]:
    try:
        # 링크는 아예 스킵
        if entry.is_symlink():
            return

        if entry.is_dir(follow_symlinks=follow_symlinks):
            stack.append(Path(entry.path))
            return

        if entry.is_file():
            st = entry.stat(follow_symlinks=follow_symlinks)
            if stat.S_ISREG(st.st_mode):
                yield Path(entry.path), st

        # 소켓/파이프/장치 등은 스킵
        return

    except (FileNotFoundError, PermissionError):
        return
    except Exception:
        return


def iter_files(target_dir: Path) -> Generator[Tuple[Path, os.stat_result]]:
    root: Path = target_dir.expanduser().resolve(strict=True)
    stack: deque[Path] = deque([root])

    while stack:
        d: Path = stack.pop()
        try:
            with os.scandir(d) as it:
                for entry in it:
                    yield from _process_entry(entry, stack)
        except (FileNotFoundError, PermissionError):
            continue
        except Exception:
            continue


In [9]:
files = iter_files(Path("~/Desktop"))
len(list(files))

1008703

In [10]:
# 병렬 처리

import os
import queue
import stat
import threading
from pathlib import Path
from typing import Generator, Tuple


def _process_directory_entry(
    entry: os.DirEntry[str],
    dir_q: queue.Queue[Path | None],
    out_q: queue.Queue[tuple[Path, os.stat_result] | None],
) -> None:
    """디렉토리 엔트리를 처리하여 파일 정보를 추출하거나 하위 디렉토리를 큐에 추가"""
    try:
        if entry.is_symlink():
            return
        if entry.is_dir(follow_symlinks=False):
            dir_q.put(Path(entry.path))
            return
        st = entry.stat(follow_symlinks=False)
        if stat.S_ISREG(st.st_mode):
            out_q.put((Path(entry.path), st))
    except (FileNotFoundError, PermissionError):
        pass


def _directory_scanner_worker(
    dir_q: queue.Queue[Path | None], out_q: queue.Queue[tuple[Path, os.stat_result] | None]
) -> None:
    """디렉토리를 스캔하여 파일들을 찾는 워커 함수"""
    while True:
        d: Path | None = dir_q.get()
        if d is None:
            dir_q.task_done()
            break
        try:
            with os.scandir(d) as it:
                for entry in it:
                    _process_directory_entry(entry, dir_q, out_q)
        except (FileNotFoundError, PermissionError):
            pass
        finally:
            dir_q.task_done()


def _shutdown_workers(
    dir_q: queue.Queue[Path | None],
    out_q: queue.Queue[tuple[Path, os.stat_result] | None],
    threads: list[threading.Thread],
    max_workers: int,
) -> None:
    """워커 스레드들을 정리하고 종료 신호를 보내는 함수"""
    dir_q.join()
    for _ in range(max_workers):
        dir_q.put(None)
    for t in threads:
        t.join()
    out_q.put(None)


def walk_files_concurrently(root: Path) -> Generator[Tuple[Path, os.stat_result]]:
    """멀티스레드를 사용하여 디렉토리 트리를 순회하며 파일들을 찾는 함수"""
    root = root.expanduser().resolve(strict=True)
    max_workers = min(32, (os.cpu_count() or 1) * 2)

    dir_q: queue.Queue[Path | None] = queue.Queue()
    out_q: queue.Queue[tuple[Path, os.stat_result] | None] = queue.Queue()

    dir_q.put(root)

    threads = [
        threading.Thread(target=_directory_scanner_worker, args=(dir_q, out_q), daemon=True)
        for _ in range(max_workers)
    ]
    for t in threads:
        t.start()

    threading.Thread(
        target=_shutdown_workers, args=(dir_q, out_q, threads, max_workers), daemon=True
    ).start()

    while True:
        item = out_q.get()
        if item is None:
            break
        yield item


In [11]:
files = walk_files_concurrently(Path("~/Desktop"))
len(list(files))


1008703

# 병렬 처리 로직 속도 비교

- 경로: "~/Desktop"
- 총 파일 개수: 1,008,703개

## 직렬: `iter_files()`

1. 25.1s
2. 25.7s
3. 26.1s

**평균: 25.6s**

## 병렬: `walk_files_concurrently()`

1. 16.7s
2. 16.6s
3. 16.4s

**평균: 16.6s**

## 결과 분석

### 시도별 성능 개선

1. -8.4s (-33.5%)
2. -9.1s (-35.4%)
3. -9.7s (-37.2%)

### 최대/최소 성능 비교

- 직렬 최대 vs 병렬 최대: 26.1s vs 16.7s = **9.4s 차이**
- 직렬 최대 vs 병렬 최소: 26.1s vs 16.4s = **9.7s 차이**
- 직렬 최소 vs 병렬 최대: 25.1s vs 16.7s = **8.4s 차이**
- 직렬 최소 vs 병렬 최소: 25.1s vs 16.4s = **8.7s 차이**

### 요약

- **평균 성능 개선**: 9.0s (35.2% 향상)
- **최대 성능 차이**: 9.7s
- **최소 성능 차이**: 8.4s
- **일관성**: 병렬 처리는 직렬 대비 안정적으로 8-10초 빠름
